# OpenOppsDB starter notebook

This public example reads the OpenOppsDB Kaggle dataset from the attached input files. It is read-only, does not need internet access, and does not use credentials.


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"

with sqlite3.connect(DB_URI, uri=True) as conn:
    tables = pd.read_sql_query(
        "select table_name, table_title, table_description "
        "from openopps_tables order by table_name",
        conn,
    )
    counts = {
        table: conn.execute(f'select count(*) from "{table}"').fetchone()[0]
        for table in tables["table_name"].tolist()
    }
    recent_jobs = pd.read_sql_query(
        """
        select
            job_id,
            title,
            company,
            locations,
            employment_type,
            first_seen_at,
            last_seen_at,
            posting_url
        from job_versions
        order by last_seen_at desc
        limit 20
        """,
        conn,
    )

summary = pd.DataFrame(
    {
        "metric": [
            "tables",
            "jobs",
            "job_versions",
            "job_sync_runs",
            "sources",
        ],
        "value": [
            counts["openopps_tables"],
            counts["jobs"],
            counts["job_versions"],
            counts["job_sync_runs"],
            counts["sources"],
        ],
    }
)
summary


In [ ]:
tables


In [ ]:
recent_jobs
